## 1) Setup & Paths

Framework used in this notebook:
- Isotropic region radius: $r=\sigma$
- Manifold axes: $a_i = \sigma\sqrt{\tilde\lambda_i}$ with $\tilde\lambda_i = \lambda_i / \lambda_{max}$
- Compare methods from saved `metrics.json` and `eigenvalues.npz`

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.certify.randomized import normalize_eigenvalues, axis_lengths
from src.indexing.base import build_index, query_index
from src.indexing.image_index import load_image_index_artifacts
from src.smoothing.pca import fit_local_pca

DATASET = "celeba"
BASE = repo_root / "output" / "smile_classification" / DATASET
CERTIFY_DIR = BASE / "certify"
PIXEL_INDEX_DIR = BASE / "index" / "pixel" / "annoy" / "euclidean"
LATENT_INDEX_DIR = BASE / "index" / "latent" / "annoy" / "euclidean"

MODES = ["pixel_isotropic", "pixel_manifold", "latent_isotropic", "latent_manifold"]
SIGMAS_ALL = ["sigma_0_25", "sigma_0_50", "sigma_0_75", "sigma_1_00"]

def sigma_val(s: str) -> float:
    return float(s.replace("sigma_", "").replace("_", \
))

def mode_label(mode: str) -> str:
    return mode.replace("_", " " ).title()

def load_metrics(folder: Path):
    if not folder.exists():
        return None
    for name in ["metrics.json", "running_metrics.json"]:
        p = folder / name
        if p.exists():
            with open(p, "r", encoding="utf-8") as f:
                data = json.load(f)
            if name == "running_metrics.json":
                return {"_status": "running", "_raw": data}
            return data
    return None

print("Setup complete")
print("repo_root:", repo_root)
print("certify_dir:", CERTIFY_DIR)
print("pixel_index_dir exists:", PIXEL_INDEX_DIR.exists())
print("latent_index_dir exists:", LATENT_INDEX_DIR.exists())

## 2) Isotropic vs Manifold (from job metrics)

In [ ]:
rows = []
for mode in MODES:
    for sig in SIGMAS_ALL:
        m = load_metrics(CERTIFY_DIR / mode / sig)
        if m is None or m.get("_status") == "running":
            continue
        rows.append({
            "mode": mode_label(mode),
            "space": mode.split("_")[0].title(),
            "smoothing": mode.split("_")[1].title(),
            "sigma": sigma_val(sig),
            "certified_acc": m.get("certified_accuracy", np.nan),
            "abstain_rate": m.get("abstain_rate", np.nan),
            "mean_radius": m.get("mean_radius", np.nan),
            "median_radius": m.get("median_radius", np.nan),
            "certified_correct": m.get("certified_correct", np.nan),
            "total": m.get("total_test_samples", np.nan),
            "smile_acc": m.get("class_smile_accuracy", np.nan),
            "no_smile_acc": m.get("class_no_smile_accuracy", np.nan),
        })

if not rows:
    print("No completed CelebA metrics found yet.")
else:
    df = pd.DataFrame(rows).sort_values(["space", "smoothing", "sigma"])
    display(df.round(4))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for space in ["Pixel", "Latent"]:
        for smooth, style in [("Isotropic", "o-"), ("Manifold", "s-")]:
            sub = df[(df["space"] == space) & (df["smoothing"] == smooth)].sort_values("sigma")
            if sub.empty:
                continue
            label = f"{space} {smooth}"
            axes[0].plot(sub["sigma"], sub["certified_acc"], style, linewidth=2, label=label)
            axes[1].plot(sub["sigma"], sub["mean_radius"], style, linewidth=2, label=label)
            axes[2].plot(sub["sigma"], sub["abstain_rate"], style, linewidth=2, label=label)

    axes[0].set_title("Certified Accuracy vs σ")
    axes[1].set_title("Mean Radius vs σ")
    axes[2].set_title("Abstention vs σ")
    axes[0].set_ylabel("Certified Accuracy")
    axes[1].set_ylabel("Mean Radius")
    axes[2].set_ylabel("Abstain Rate")
    for ax in axes:
        ax.set_xlabel("σ")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=9)
    plt.suptitle("CelebA Results from Jobs", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

## 3) Same Accuracy Matching (Manifold vs Isotropic)

In [ ]:
if "df" not in globals() or df.empty:
    print("Run Section 2 first.")
else:
    all_matches = []
    for space in ["Pixel", "Latent"]:
        mani = df[(df["space"] == space) & (df["smoothing"] == "Manifold")].sort_values("sigma")
        iso = df[(df["space"] == space) & (df["smoothing"] == "Isotropic")].sort_values("sigma")
        if mani.empty or iso.empty:
            continue
        for _, mrow in mani.iterrows():
            idx = (iso["certified_acc"] - mrow["certified_acc"]).abs().idxmin()
            irow = iso.loc[idx]
            all_matches.append({
                "space": space,
                "target_acc": float(mrow["certified_acc"]),
                "sigma_mani": float(mrow["sigma"]),
                "sigma_iso_closest": float(irow["sigma"]),
                "abs_gap": float(abs(mrow["certified_acc"] - irow["certified_acc"])),
            })

    if not all_matches:
        print("No manifold/isotropic pairs found for matching.")
    else:
        df_match = pd.DataFrame(all_matches).sort_values(["space", "target_acc"], ascending=[True, False])
        display(df_match.round(4))

        fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
        for ax, space in zip(axes, ["Pixel", "Latent"]):
            sub = df_match[df_match["space"] == space]
            if sub.empty:
                ax.set_title(f"{space}: no matches")
                ax.grid(True, alpha=0.3)
                continue
            ax.plot(sub["target_acc"], sub["sigma_mani"], "o-", label="σ_mani")
            ax.plot(sub["target_acc"], sub["sigma_iso_closest"], "s-", label="σ_iso closest")
            ax.set_title(f"{space} Space")
            ax.set_xlabel("Target Certified Accuracy")
            ax.grid(True, alpha=0.3)
            ax.legend()
        axes[0].set_ylabel("Required σ")
        plt.suptitle("Same-Accuracy Matching: Required σ", fontsize=13, y=1.02)
        plt.tight_layout()
        plt.show()

## 4) New Volume Geometry Metrics from `metrics.json`

Reads `volume.geometry` when present, with fallback to legacy `volume` keys.

In [ ]:
vol_rows = []
for mode in ["pixel_manifold", "latent_manifold"]:
    for sig in SIGMAS_ALL:
        m = load_metrics(CERTIFY_DIR / mode / sig)
        if m is None or m.get("_status") == "running":
            continue
        vol = m.get("volume", {}) if isinstance(m, dict) else {}
        geo = vol.get("geometry", {}) if isinstance(vol, dict) else {}
        vol_rows.append({
            "mode": mode_label(mode),
            "space": mode.split("_")[0].title(),
            "sigma": sigma_val(sig),
            "mean_log_geo_ratio": geo.get("mean_log_geo_ratio", vol.get("mean_geometry_factor", np.nan)),
            "mean_anisotropy_ratio": geo.get("mean_anisotropy_ratio", np.nan),
            "mean_effective_rank": geo.get("mean_effective_rank", vol.get("mean_effective_rank", np.nan)),
            "log_v_iso_geo": geo.get("log_v_iso_geo", np.nan),
            "mean_log_v_mani_geo": geo.get("mean_log_v_mani_geo_max_norm", geo.get("mean_log_v_mani_geo", np.nan)),
        })

if not vol_rows:
    print("No volume geometry metrics found.")
else:
    df_geo = pd.DataFrame(vol_rows).sort_values(["space", "sigma"])
    display(df_geo.round(4))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for space in ["Pixel", "Latent"]:
        sub = df_geo[df_geo["space"] == space].sort_values("sigma")
        if sub.empty:
            continue
        axes[0].plot(sub["sigma"], sub["mean_log_geo_ratio"], "o-", linewidth=2, label=space)
        axes[1].plot(sub["sigma"], sub["mean_anisotropy_ratio"], "o-", linewidth=2, label=space)
        axes[2].plot(sub["sigma"], sub["mean_effective_rank"], "o-", linewidth=2, label=space)

    axes[0].set_title("Geometry Gain: log(V_mani_geo / V_iso_geo)")
    axes[1].set_title("Mean Anisotropy Ratio")
    axes[2].set_title("Mean Effective Rank")
    axes[0].set_ylabel("mean log geo ratio")
    axes[1].set_ylabel("a1/ak")
    axes[2].set_ylabel("effective rank")
    for ax in axes:
        ax.set_xlabel("σ")
        ax.grid(True, alpha=0.3)
        ax.legend()

    plt.suptitle("CelebA New Volume Geometry Metrics", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

## 5) Full Eigen / Axis Spectrum from `eigenvalues.npz`

Reads full arrays such as `axis_lengths_all`, with fallback to legacy keys.

In [ ]:
TARGET_SIGMA = "sigma_0_50"
for mode in ["pixel_manifold", "latent_manifold"]:
    eigen_path = CERTIFY_DIR / mode / TARGET_SIGMA / "eigenvalues.npz"
    if not eigen_path.exists():
        print(f"No eigenvalues file at {eigen_path}")
        continue

    data = np.load(eigen_path)
    axis_arr = data.get("axis_lengths_all", data.get("axis_lengths_top10", None))
    anis = data.get("anisotropy_ratios", None)
    geo_ratio = data.get("log_geo_ratio_per_sample", data.get("log_geo_ratio_per_token", None))

    if axis_arr is None:
        print(f"Axis arrays not found in {eigen_path}")
        continue

    axis_arr = np.asarray(axis_arr, dtype=np.float64)
    mean_axes = np.nanmean(axis_arr, axis=0)
    cum = np.cumsum(np.maximum(mean_axes, 0.0))
    if len(cum) > 0 and cum[-1] > 0:
        cum = cum / cum[-1]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].plot(np.arange(1, len(mean_axes) + 1), mean_axes, "o-", linewidth=2, markersize=4)
    axes[0].set_title(f"Mean Axis Spectrum ({mode})")
    axes[0].set_xlabel("component i")
    axes[0].set_ylabel("mean a_i")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(np.arange(1, len(cum) + 1), cum, "o-", linewidth=2, markersize=4, color="tab:green")
    axes[1].axhline(0.9, linestyle="--", color="orange", alpha=0.8)
    axes[1].axhline(0.95, linestyle="--", color="red", alpha=0.8)
    axes[1].set_title("Cumulative Axis Contribution")
    axes[1].set_xlabel("top components")
    axes[1].set_ylabel("cumulative share")
    axes[1].grid(True, alpha=0.3)

    if anis is not None:
        axes[2].hist(np.asarray(anis, dtype=np.float64), bins=30, color="tab:purple", alpha=0.75, edgecolor="black")
        axes[2].set_title("Anisotropy Distribution")
        axes[2].set_xlabel("a1/ak")
    elif geo_ratio is not None:
        axes[2].hist(np.asarray(geo_ratio, dtype=np.float64), bins=30, color="tab:red", alpha=0.75, edgecolor="black")
        axes[2].set_title("Geometry Gain Distribution")
        axes[2].set_xlabel("log geo ratio")
    else:
        axes[2].text(0.5, 0.5, "No anisotropy/geo arrays", ha="center", va="center")
    axes[2].set_ylabel("count")
    axes[2].grid(True, alpha=0.3)

    plt.suptitle(f"Eigen / Axis Analysis ({mode}, σ={sigma_val(TARGET_SIGMA)})", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

    print("Loaded:", eigen_path)
    print("axis_arr shape:", axis_arr.shape)

## 6) Embedding Neighborhood: Circle vs Ellipse

Local PCA geometry interpretation on index vectors:
- Blue circle: isotropic region, radius $r=\sigma$
- Green ellipse: manifold region with axes $a_i = \sigma\sqrt{\tilde\lambda_i}$

In [ ]:
SPACE = "latent"  # "pixel" or "latent"
sigma = 0.50
knn_k = 300
anchor_idx = 0

index_dir = LATENT_INDEX_DIR if SPACE == "latent" else PIXEL_INDEX_DIR
if not index_dir.exists():
    raise FileNotFoundError(f"Index directory not found: {index_dir}")

vectors, meta = load_image_index_artifacts(index_dir)
idx = build_index(vectors=vectors, backend="torch", metric="euclidean")

anchor_idx = min(anchor_idx, len(vectors) - 1)
anchor_vec = vectors[anchor_idx]
neighbor_ids = query_index(idx, k=min(knn_k + 1, len(vectors)), vector=anchor_vec)[1:]
neighbor_vecs = vectors[neighbor_ids]

pca = fit_local_pca(neighbor_vecs)
evals = np.asarray(pca.evals, dtype=np.float64)
evals_norm = normalize_eigenvalues(evals, mode="max")
axes_len = axis_lengths(sigma, evals_norm)
if len(axes_len) < 2:
    raise ValueError("Need at least 2 PCA components for visualization.")

all_pts = np.vstack([anchor_vec.reshape(1, -1), neighbor_vecs])
proj2 = (all_pts - pca.mean) @ pca.evecs[:, :2]
anchor_2d = proj2[0]
nb_2d = proj2[1:]

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(nb_2d[:, 0], nb_2d[:, 1], c="lightgray", s=18, alpha=0.45, label="kNN points")
ax.scatter(anchor_2d[0], anchor_2d[1], c="gold", s=260, marker="*", edgecolors="black", label=f"Anchor idx={anchor_idx}")

circle = plt.Circle((anchor_2d[0], anchor_2d[1]), sigma, fill=False, color="blue", linewidth=2.5,
                    label=f"Iso circle (r=σ={sigma})")
ax.add_patch(circle)

ellipse = Ellipse((anchor_2d[0], anchor_2d[1]), width=2 * axes_len[0], height=2 * axes_len[1],
                  fill=False, color="green", linewidth=2.5,
                  label=f"Mani ellipse (a1={axes_len[0]:.3f}, a2={axes_len[1]:.3f})")
ax.add_patch(ellipse)

ax.set_title(f"CelebA {SPACE.title()} Neighborhood: Circle vs Ellipse")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.grid(True, alpha=0.25)
ax.set_aspect("equal")
ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

print("Interpretation:")
print("- Blue circle = isotropic smoothing region.")
print("- Green ellipse = manifold-shaped region from local eigen-geometry.")
print("- Axes follow a_i = sigma * sqrt(lambda_tilde_i).")